# LoCI LAMP Product Creation - Production Phase

This notebook completes the LoCI LAMP product creation with Digital Product Passport (DPP).

**Flow mirrors the GUI CreateProjectForm.tsx onSubmit:**
1. Load setup data and components
2. Create DPP structure from CSV data
3. Submit DPP to interfacer-dpp service (gets ULID)
4. Produce the final LOCI LAMP with DPP metadata
5. Trace and verify the complete supply chain

In [41]:
# Module imports and auto-reload setup
%load_ext autoreload
%aimport if_lib, if_utils, if_dpp, if_graphics, if_consts, if_gc1dpp
%autoreload 1
import os
import json
import random

from if_utils import get_filename, show_data, save_traces

from if_lib import generate_random_challenge, read_HMAC, read_keypair, get_id_person, get_location_id, \
get_unit_id, get_resource_spec_id, get_resource, get_process, create_event, make_transfer, reduce_resource, set_user_location, send, send_signed

from if_dpp import trace_query, check_traces, er_before, get_dpp

from if_graphics import vis_dpp, make_sankey, consol_trace

from if_gc1dpp import submit_dpp, upload_file_on_dpp

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Configuration and Endpoints

In [ ]:
# Define constants - must match the setup notebook
USE_CASE = 'locilamp'

# Zenflows API endpoint
ENDPOINT = 'https://proxy.dpp-staging.dnstest.dyne.org/zenflows/api'

# DPP service endpoint
DPP_URL = 'https://proxy.dpp-staging.dnstest.dyne.org/interfacer-dpp'

# Participants
USERS = ['tchibo', 'locilamp_designer', 'locilamp_manufacturer']

## Load Setup Data from JSON Files

In [43]:
# Define file paths for saved data from setup notebook
# Use get_filename to match the setup notebook's pattern
users_file = get_filename('cred_users.json', ENDPOINT, USE_CASE)
locations_file = get_filename('loc_users.json', ENDPOINT, USE_CASE)
units_file = get_filename('units_data.json', ENDPOINT, USE_CASE)
specs_file = get_filename('res_spec_data.json', ENDPOINT, USE_CASE)
processes_file = get_filename('process_data.json', ENDPOINT, USE_CASE)
resources_file = get_filename('initial_resources.json', ENDPOINT, USE_CASE)
images_file = get_filename('images_data.json', ENDPOINT, USE_CASE)
product_file = get_filename('product_data.json', ENDPOINT, USE_CASE)

# NEW: Files for GUI-aligned resources
materials_file = get_filename('material_resources.json', ENDPOINT, USE_CASE)
design_file = get_filename('design_resource.json', ENDPOINT, USE_CASE)
gui_specs_file = get_filename('gui_specs.json', ENDPOINT, USE_CASE)

# Load all data
with open(users_file) as f:
    users_data = json.load(f)
    
with open(locations_file) as f:
    locations_data = json.load(f)
    
with open(units_file) as f:
    units_data = json.load(f)
    
with open(specs_file) as f:
    specs_data = json.load(f)
    
with open(processes_file) as f:
    processes_data = json.load(f)
    
with open(resources_file) as f:
    resources_data = json.load(f)
    
with open(images_file) as f:
    images_data = json.load(f)
    
with open(product_file) as f:
    product_data = json.load(f)

# NEW: Load GUI-aligned resources
try:
    with open(materials_file) as f:
        material_resources = json.load(f)
except FileNotFoundError:
    material_resources = {}
    print("⚠️ No material_resources.json found - run Setup notebook first")

try:
    with open(design_file) as f:
        design_resource = json.load(f)
except FileNotFoundError:
    design_resource = {}
    print("⚠️ No design_resource.json found - run Setup notebook first")

try:
    with open(gui_specs_file) as f:
        gui_specs = json.load(f)
except FileNotFoundError:
    gui_specs = {}
    print("⚠️ No gui_specs.json found - run Setup notebook first")

print("✅ All setup data loaded successfully!")
print(f"   Users: {len(users_data)}")
print(f"   Locations: {len(locations_data)}")
print(f"   Units: {len(units_data)}")
print(f"   Resource Specs: {len(specs_data)}")
print(f"   Processes: {len(processes_data)}")
print(f"   Resources: {len(resources_data)}")
print(f"   Uploaded Images: {len(images_data)}")
print(f"   Product Data loaded from CSV")
print(f"\n📋 GUI-aligned resources:")
print(f"   Material Resources: {len(material_resources)}")
print(f"   Design Resource: {'Yes' if design_resource.get('id') else 'No'}")
print(f"   GUI Specs: {len([k for k, v in gui_specs.items() if v])}")

✅ All setup data loaded successfully!
   Users: 3
   Locations: 3
   Units: 4
   Resource Specs: 15
   Processes: 5
   Resources: 10
   Uploaded Images: 4
   Product Data loaded from CSV

📋 GUI-aligned resources:
   Material Resources: 6
   Design Resource: Yes
   GUI Specs: 7


## Verify Components Available for Production

In [44]:
# Verify that we have all required components available
# These are the components that were created and transferred to Tchibo

tchibo_user = users_data.get("tchibo", {})
print(f"Production user: {tchibo_user.get('name', 'Unknown')}")
print(f"User ID: {tchibo_user.get('id', 'Unknown')}")

# List available resources for production
print("\n📦 Components available for production:")
for res_name, res_info in resources_data.items():
    print(f"   - {res_name}: {res_info.get('name', 'Unknown')}")

# Get the main resource IDs we'll need
assembled_lamp_spec = specs_data.get("loci_lamp_assembled", {})
print(f"\n🔧 Target product specification: {assembled_lamp_spec.get('name', 'Unknown')}")
print(f"   Spec ID: {assembled_lamp_spec.get('id', 'Unknown')}")

Production user: Tchibo GmbH
User ID: 06EG0S695QJ78QRCP79B4EYCSC

📦 Components available for production:
   - kroma_kraft_cardboard: kroma_kraft_cardboard
   - acidfree_paper: acidfree_paper
   - textile_cable: textile_cable
   - e27_socket: e27_socket
   - eu_plug: eu_plug
   - toggle_switch: toggle_switch
   - locilamp_cardboard_components: LOCI LAMP cardboard lamelles and base
   - locilamp_lampshade: LOCI LAMP transparent lampshade
   - locilamp_electrical_assembly: LOCI LAMP electrical assembly
   - locilamp_design: LOCI LAMP V 2.0 design

🔧 Target product specification: Unknown
   Spec ID: Unknown


## Create DPP Structure from LOCI LAMP Data

Build the Digital Product Passport structure using the data from the CSV file.

In [45]:
# Use the product data loaded from CSV (saved by setup notebook)
# This mirrors the data that would come from CreateProjectForm.tsx in the GUI

loci_lamp_data = product_data

print("📋 LOCI LAMP DPP Data prepared from CSV:")
print(f"   Product: {loci_lamp_data.get('Product Overview', {}).get('Product Name', 'Unknown')}")
po = loci_lamp_data.get('Product Overview', {})
print(f"   Brand: {po.get('Brand Name', 'N/A')}")
print(f"   Model: {po.get('Model Name', 'N/A')}")
print(f"   Images: {len(images_data)} uploaded")

📋 LOCI LAMP DPP Data prepared from CSV:
   Product: LOCI LAMP
   Brand: Tchibo GmbH
   Model: LOCILAMP V 2.0
   Images: 4 uploaded


## Submit DPP to interfacer-dpp Service

Submit the Digital Product Passport to the DPP service and receive a ULID (Universally Unique Lexicographically Sortable Identifier).

**Temporary workaround:** Filter out failed image-upload entries from the payload. Remove this after the interfacer-dpp update.

In [46]:
# Force reload of if_gc1dpp to pick up latest changes
import importlib
import if_gc1dpp
importlib.reload(if_gc1dpp)
from if_gc1dpp import submit_dpp, upload_file_on_dpp, process_dpp_values
print("✓ if_gc1dpp module reloaded")

✓ if_gc1dpp module reloaded


In [47]:
# Get Tchibo's credentials and person ID for DPP submission
# Extract credentials from users_data (loaded from setup notebook)
from datetime import datetime

user_name = "tchibo"
user_for_dpp = users_data[user_name]

# Get EdDSA keys for signing
eddsa_public_key = user_for_dpp['eddsa_public_key']
eddsa_private_key = user_for_dpp['keyring']['eddsa']
eddsa_keys = {
    'public_key': eddsa_public_key,
    'private_key': eddsa_private_key
}

# Get person ID
tchibo_id = user_for_dpp['id']
print(f"Tchibo user ID: {tchibo_id}")
print(f"Using credentials for: {user_for_dpp['name']}")

# The user_for_dpp dict contains the auth data for API calls
# Store it for use in subsequent cells
HMAC = user_for_dpp  # Pass the whole user object, functions expect this

# Build the DPP payload from the CSV product data
# This mirrors the processDppValues + dpp schema structure from CreateProjectForm.tsx
po = loci_lamp_data.get('Product Overview', {})
repairability = loci_lamp_data.get('Repairability', {})
env_impact = loci_lamp_data.get('Environmental Impact', {})
compliance = loci_lamp_data.get('Compliance and Standards', {})
certificates = loci_lamp_data.get('Certificates', {})
recyclability = loci_lamp_data.get('Recyclability', {})
energy = loci_lamp_data.get('Energy Use & Efficiency', {})
component_info = loci_lamp_data.get('Component Information Drill', {})
economic_operator = loci_lamp_data.get('Economic Operator', {})

# Temporary workaround: filter out failed image uploads (auth errors/log payloads)
# Remove this after interfacer-dpp update.

def _valid_image_payload(img: dict) -> bool:
    if not isinstance(img, dict):
        return False
    if img.get('error'):
        return False
    return bool(img.get('id') or img.get('url'))

images_clean = {k: v for k, v in images_data.items() if _valid_image_payload(v)}
image_list = list(images_clean.values())

# Helper to build DPP transformed values (GUI schema)
def _tv(value, type_="Text", units=None):
    if value is None:
        return None
    if isinstance(value, str) and not value.strip():
        return None
    if units is not None:
        return {"type": type_, "value": value, "units": units}
    return {"type": type_, "value": value}

def _prune(obj):
    if isinstance(obj, dict):
        out = {}
        for k, v in obj.items():
            pv = _prune(v)
            if pv is None:
                continue
            if isinstance(pv, dict) and not pv:
                continue
            if isinstance(pv, list) and not pv:
                continue
            out[k] = pv
        return out
    if isinstance(obj, list):
        items = [_prune(v) for v in obj]
        items = [v for v in items if v is not None and (not isinstance(v, dict) or v) and (not isinstance(v, list) or v)]
        return items if items else None
    return obj

# Build the GUI-compatible DPP payload
raw_dpp_payload = {
    "productOverview": {
        "brandName": _tv(po.get('Brand Name')),
        "countryOfSale": _tv(po.get('Country of Sale')),
        "productDescription": _tv(po.get('Product Description')),
        "productName": _tv(po.get('Product Name')),
        "netWeight": _tv(po.get('Net Weight')),
        "color": _tv(po.get('Color')),
        "countryOfOrigin": _tv(po.get('Country of Origin')),
        "dimensions": _tv(po.get('Dimensions')),
        "modelName": _tv(po.get('Model Name')),
        "conditionOfTheProduct": _tv(po.get('Condition of the Product')),
        "netContent": _tv(po.get('Net Content')),
        "safetyInstructions": _tv(po.get('Safety Instructions')),
        "gtin": _tv(po.get('GTIN')),
        "productImage": _tv(image_list if image_list else None)
    },
    "reparability": {
        "serviceAndRepairInstructions": _tv(repairability.get('Service and Repair Instructions')),
        "availabilityOfSpareParts": _tv(repairability.get('Availability of Spare Parts'))
    },
    "environmentalImpact": {
        "co2eEmissionsPerUnit": _tv(env_impact.get('CO₂e Emissions per Unit') or env_impact.get('CO2e Emissions per Unit')),
        "energyConsumptionPerUnit": _tv(env_impact.get('Energy Consumption per Unit')),
        "waterConsumptionPerUnit": _tv(env_impact.get('Water Consumption per Unit')),
        "chemicalConsumptionPerUnit": _tv(env_impact.get('Chemical Consumption per Unit')),
        "minimumContentOfMaterialWithSustainabilityCertification": _tv(env_impact.get('Minimum Content of Material with Sustainability Certification')),
        "cleaningPerformanceAtLowTemperature": _tv(env_impact.get('Cleaning Performance at Low Temperature'))
    },
    "complianceAndStandards": {
        "ceMarking": _tv(compliance.get('CE Marking')),
        "rohsCompliance": _tv(compliance.get('RoHS Compliance'))
    },
    "certificates": {
        "nameOfCertificate": _tv(certificates.get('Name of Certificate'))
    },
    "recyclability": {
        "recyclingInstructions": _tv(recyclability.get('Recycling Instructions')),
        "materialComposition": _tv(recyclability.get('Material Composition')),
        "substancesOfConcern": _tv(recyclability.get('Substances of Concern'))
    },
    "energyUseAndEfficiency": {
        "maximumElectricalPower": _tv(energy.get('Maximum Electrical Power')),
        "maximumVoltage": _tv(energy.get('Maximum Voltage')),
        "maximumCurrent": _tv(energy.get('Maximum Current')),
        "powerRating": _tv(energy.get('Power Rating')),
        "dcVoltage": _tv(energy.get('DC Voltage')),
        "batteryType": _tv(energy.get('Battery Type')),
        "batteryChargingTime": _tv(energy.get('Battery Charging Time')),
        "batteryLife": _tv(energy.get('Battery Life')),
        "chargerType": _tv(energy.get('Charger Type'))
    },
    "components": [
        {
            "componentDescription": _tv(component_info.get('Component Description')),
            "componentGTIN": _tv(component_info.get('Component GTIN')),
            "linkToDPP": _tv(component_info.get('Link to DPP'))
        }
    ],
    "economicOperator": {
        "companyName": _tv(economic_operator.get('Company name')),
        "addressLine1": _tv(economic_operator.get('Address line 1 (street & house number)')),
        "addressLine2": _tv(economic_operator.get('Address line 2 (postal code & city)')),
        "contactInformation": _tv(economic_operator.get('contact information (email)')),
        "gln": _tv(economic_operator.get('GLN')),
        "eoriNumber": _tv(economic_operator.get('EORI Number'))
    }
}

dpp_payload = _prune(raw_dpp_payload) or {}

print("\n📤 DPP payload prepared for submission")
print(f"   Product: {po.get('Product Name', 'LOCI LAMP')}")
print(f"   Created by: {user_for_dpp['name']}")
print(f"   Images included: {len(image_list)}")

Tchibo user ID: 06EG0S695QJ78QRCP79B4EYCSC
Using credentials for: Tchibo GmbH

📤 DPP payload prepared for submission
   Product: LOCI LAMP
   Created by: Tchibo GmbH
   Images included: 4


In [48]:
# Submit DPP to the interfacer-dpp service
# This is the equivalent of signedPost to DPP_URL/dpp in CreateProjectForm.tsx

# Use the reloaded module directly
import if_gc1dpp
submit_dpp_func = if_gc1dpp.submit_dpp

try:
    dpp_ulid = submit_dpp_func(dpp_payload, eddsa_public_key, eddsa_private_key, DPP_URL)

    print(f"✅ DPP submitted successfully!")
    print(f"   ULID: {dpp_ulid}")
    print(f"   DPP URL: {DPP_URL}/dpp/{dpp_ulid}")
except Exception as e:
    print(f"❌ Error submitting DPP: {e}")
    import traceback
    traceback.print_exc()
    print("Note: Make sure the DPP service is running at", DPP_URL)
    dpp_ulid = None

Submitting DPP to https://proxy.dpp-staging-tchibo.dnstest.dyne.org/interfacer-dpp/dpp
Public key: 3hpGAbEpBNas1pjua21c...
Signature: jrCkXN2aZ3Jy0WcqbkT6cCeY74a4ZWr1l3KBnkyL...
Response status: 200
Response text: {"insertedID":"01KM07CJ7088DMVGWC28X0S1VJ"}
DPP submitted with ULID: 01KM07CJ7088DMVGWC28X0S1VJ
✅ DPP submitted successfully!
   ULID: 01KM07CJ7088DMVGWC28X0S1VJ
   DPP URL: https://proxy.dpp-staging-tchibo.dnstest.dyne.org/interfacer-dpp/dpp/01KM07CJ7088DMVGWC28X0S1VJ


## Produce LOCI LAMP with DPP Metadata

Create the final product in Zenflows, linking it to the DPP ULID. This mirrors the `handleProjectCreation` function from CreateProjectForm.tsx.

In [49]:
# Get or create the assembly process for Tchibo
process_name = "Create_locilamp_assembly"
process_note = "Assemble LOCI LAMP from components"

if process_name not in processes_data:
    get_process(process_name, processes_data, process_note, users_data["tchibo"], endpoint=ENDPOINT)

assembly_process = processes_data.get(process_name, {})
assembly_process_id = assembly_process.get("id")
print(f"Assembly process ID: {assembly_process_id}")

# Get location and unit IDs
berlin_location = locations_data.get("berlin", {})
unit_each = units_data.get("piece", {})

# Fetch product and DPP resource spec IDs from backend instanceVariables
SPEC_PROJECT_PRODUCT_ID = gui_specs.get("specProjectProduct", {}).get("id") if gui_specs else None
SPEC_DPP_ID = gui_specs.get("specDpp", {}).get("id") if gui_specs else None
instance_specs_json = None

try:
    specs_query = """
    query {
        instanceVariables {
            specs {
                specProjectProduct { id name }
                specDpp { id name }
            }
        }
    }
    """

    try:
        instance_specs_json = send_signed(
            specs_query,
            {},
            user_for_dpp["username"],
            user_for_dpp["keyring"]["eddsa"],
            ENDPOINT
        )
    except Exception as e:
        print(f"⚠️ Signed query failed, trying unsigned: {e}")

    if not instance_specs_json:
        specs_res = send(payload={"query": specs_query, "variables": {}}, endpoint=ENDPOINT)
        instance_specs_json = specs_res.json()

    specs_payload = instance_specs_json.get("data", {}).get("instanceVariables", {}).get("specs", {})
    SPEC_PROJECT_PRODUCT_ID = specs_payload.get("specProjectProduct", {}).get("id") or SPEC_PROJECT_PRODUCT_ID
    SPEC_DPP_ID = specs_payload.get("specDpp", {}).get("id") or SPEC_DPP_ID
except Exception as e:
    print(f"⚠️ Could not fetch instanceVariables specs: {e}")

assembled_spec_id = (
    SPEC_PROJECT_PRODUCT_ID
    or specs_data.get("locilamp_assembled", {}).get("id")
    or assembled_lamp_spec.get("id")
    or gui_specs.get("assembledProductId")
    if gui_specs else None
)

if not assembled_spec_id:
    raise ValueError("Product resource specification ID is unavailable from backend and setup data")

# Ensure specs_data contains the product spec with default unit
if "locilamp_assembled" not in specs_data:
    specs_data["locilamp_assembled"] = {
        "id": assembled_spec_id,
        "name": "locilamp_assembled",
        "defaultUnit": unit_each.get("id")
    }

print(f"Assembled lamp spec ID: {assembled_spec_id}")
print(f"DPP spec ID: {SPEC_DPP_ID}")
print(f"Location: {berlin_location.get('name')}")
print(f"Unit: {unit_each.get('label')}")

Assembly process ID: 06EG0XJASTZG3RXPNZKRPRPY60
Assembled lamp spec ID: 06E7CCNH2BVJ4NGT6MA0K4MXGR
DPP spec ID: 06E7CCNH56YC5K5ERRMAPFQFHG
Location: None
Unit: u_piece


In [50]:
# Consume the component resources (consume action)
# In the setup, components were transferred to Tchibo, now we consume them for assembly

consumed_resources = []

# Define which resources to consume (names from setup notebook)
components_to_use = [
    'locilamp_cardboard_components',
    'locilamp_lampshade',
    'locilamp_electrical_assembly'
]

for component_name in components_to_use:
    if component_name in resources_data:
        cur_res = resources_data[component_name]
        action = 'consume'
        event_note = f"consume {component_name} for LOCI LAMP assembly"
        amount = 1

        event_id, ts = create_event(
            users_data['tchibo'],
            action,
            event_note,
            amount=amount,
            process=assembly_process,
            res_spec_data=specs_data,
            existing_res=cur_res,
            endpoint=ENDPOINT
        )
        consumed_resources.append({
            "name": component_name,
            "event_id": event_id,
            "resource_id": cur_res.get('id')
        })
        print(f"✅ Consumed: {component_name}")

print(f"\n📦 Total components consumed: {len(consumed_resources)}")

✅ Consumed: locilamp_cardboard_components
✅ Consumed: locilamp_lampshade
✅ Consumed: locilamp_electrical_assembly

📦 Total components consumed: 3


In [51]:
# ============================================================================
# PRODUCE LOCI LAMP - MATCHING GUI CreateProjectForm.tsx + useProjectCRUD.ts
# ============================================================================
# This cell implements the exact flow from:
#   1. CREATE_PROJECT mutation with all form fields
#   2. CREATE_DPP_RESOURCE mutation with metadata={dppServiceUlid}
#   3. CITE_PROJECT mutation to link DPP resource to product process
#   4. CITE design resource (LinkDesignStep)
#   5. CONSUME materials (MaterialsStep)
#   6. USE machines (MachinesStep)
#   7. CITE relations (RelationsStep)
# ============================================================================

from datetime import datetime
import requests

# ============================================================================
# STEP 1: Prepare all form data (mirrors CreateProjectValues interface)
# ============================================================================

# MainStep values - REQUIRED
main_title = po.get('Product Name', 'LOCI LAMP')

# Build a proper description - CSV Product Description might be too short
csv_description = po.get('Product Description', '')
net_content = po.get('Net Content', '')

# Use Net Content if Product Description is too short (less than 50 chars)
if len(csv_description) < 50 and net_content:
    main_description = f"{csv_description} - {net_content}"
elif len(csv_description) < 50:
    # Fallback to a detailed description
    main_description = (
        'Tischleuchte - Table lamp from sustainable materials. '
        'Self-assembly kit with FSC-certified cardboard and acid-free paper. '
        'Designed for easy repair and recycling.'
    )
else:
    main_description = csv_description

main_link = "https://github.com/locilamp/locilamp-v2"  # repo URL (REQUIRED by GUI)
main_tags = [
    "lamp",
    "sustainable", 
    "cardboard",
    "FSC",
    "DIY",
    "table-lamp",
    "design",
    "Tchibo"
]

# LocationStep values (from locations_data loaded from Setup)
location_name = locations_data.get("tchibo", {}).get("name", "Tchibo GmbH Headquarters")
location_address = locations_data.get("tchibo", {}).get("addr", "Überseering 18, 22297 Hamburg, Germany")
location_lat = locations_data.get("tchibo", {}).get("lat", 53.6040379)
location_long = locations_data.get("tchibo", {}).get("long", 10.0221277)
location_remote = False

# LicenseStep values
licenses = [
    {"scope": "hardware", "licenseId": "CC-BY-SA-4.0"},
    {"scope": "documentation", "licenseId": "CC-BY-4.0"}
]

# ContributorsStep values (agent ULIDs)
contributors = [
    users_data.get('locilamp_designer', {}).get('id'),  # Designer
    users_data.get('locilamp_manufacturer', {}).get('id')  # Manufacturer
]
contributors = [c for c in contributors if c]  # Remove None values

# RelationsStep values (related/included resource IDs)
# Include the semi-finished components as relations
relations = []
for comp_name in ['locilamp_cardboard_components', 'locilamp_lampshade', 'locilamp_electrical_assembly']:
    if comp_name in resources_data and resources_data[comp_name].get('id'):
        relations.append(resources_data[comp_name]['id'])

# DeclarationsStep values (REQUIRED for Product type) - matches GUI schema
declarations = {
    "repairable": "yes",  # Is product repairable?
    "recyclable": "yes",  # Is product recyclable?
    "certifications": [
        {"url": "https://fsc.org/", "label": "FSC Certified Paper and Cardboard"},
        {"url": "https://www.tuvsud.com/", "label": "TÜV Electrical Safety"}
    ]
}

# ProductFiltersStep values - from CSV data
product_filters = {
    "categories": ["lighting", "home-decor", "sustainable"],
    "powerCompatibility": ["220-240V AC", "50Hz"],
    "replicability": ["partial"],  # Can be partially replicated
    "powerRequirementW": int(po.get('Wattage', '8').replace('W', '').strip() or 8),
    "energyKwh": float(energy.get('Energy Consumption per Unit', '0.008') or 0.008),
    "co2Kg": float(env_impact.get('CO2e Emissions per Unit', '1.75') or 1.75)
}

# MachinesStep values - machines needed for production
machines_used = []  # Machine resource IDs (if any)
machine_details = [
    {"id": None, "name": "Laser Cutter"},  # For cardboard cutting
    {"id": None, "name": "Paper Cutter"}   # For lampshade
]

# MaterialsStep values - from material_resources loaded from Setup
# These are the materials with specMaterial that will be CONSUMED
materials_used = []  # Material resource IDs
material_details = []  # Material metadata for tags

for mat_name, mat_data in material_resources.items():
    if mat_data.get('id'):
        materials_used.append(mat_data['id'])
        material_details.append({
            "id": mat_data['id'],
            "name": mat_name
        })

# If no materials from Setup, use fallback names
if not material_details:
    material_details = [
        {"id": None, "name": "KROMA KRAFT Displaykarton (FSC)"},
        {"id": None, "name": "Säurefreies Papier (FSC)"},
        {"id": None, "name": "Textilkabel H03VVH2-F"},
        {"id": None, "name": "E27 Lampenfassung"},
        {"id": None, "name": "EU Stecker (2-PIN)"},
        {"id": None, "name": "Kippschalter"}
    ]

# LinkedDesign (REQUIRED for Product type) - from design_resource loaded from Setup
linked_design = design_resource.get('id') if design_resource else gui_specs.get('designResourceId')

print(f"📋 Form Data Prepared (matching GUI CreateProjectValues):")
print(f"   Title: {main_title}")
print(f"   Description: {len(main_description)} chars")
print(f"   Repo URL: {main_link}")
print(f"   Tags: {len(main_tags)}")
print(f"   Location: {location_name}")
print(f"   Licenses: {len(licenses)}")
print(f"   Contributors: {len(contributors)}")
print(f"   Relations: {len(relations)}")
print(f"   Declarations: repairable={declarations['repairable']}, recyclable={declarations['recyclable']}")
print(f"   Materials: {len(material_details)}")
print(f"   Machines: {len(machine_details)}")
print(f"   Linked Design: {linked_design or 'None'}")

# ============================================================================
# STEP 2: Build classifiedAs tags (matches GUI derivedProductFilterTags + mergeTags)
# ============================================================================

def build_classified_as_tags(base_tags, product_filters, machine_details, material_details):
    """Build classifiedAs array matching GUI tagging.ts pattern."""
    tags = list(base_tags)
    
    # Add machine tags with prefix
    for m in machine_details:
        if m.get("name"):
            tags.append(f"m:{m['name']}")
    
    # Add material tags with prefix
    for mat in material_details:
        if mat.get("name"):
            tags.append(f"mat:{mat['name']}")
    
    # Add category tags
    for cat in product_filters.get("categories", []):
        tags.append(f"c:{cat}")
    
    # Add power compatibility tags
    for pc in product_filters.get("powerCompatibility", []):
        tags.append(f"pc:{pc}")
    
    # Add replicability tags
    for rep in product_filters.get("replicability", []):
        tags.append(f"rep:{rep}")
    
    # Add numeric filter tags
    if product_filters.get("powerRequirementW") is not None:
        tags.append(f"pwr:{product_filters['powerRequirementW']}")
    if product_filters.get("energyKwh") is not None:
        tags.append(f"env:energy:{product_filters['energyKwh']}")
    if product_filters.get("co2Kg") is not None:
        tags.append(f"env:co2:{product_filters['co2Kg']}")
    
    return tags

classified_as_tags = build_classified_as_tags(main_tags, product_filters, machine_details, material_details)
print(f"📌 ClassifiedAs tags: {len(classified_as_tags)} tags")
for t in classified_as_tags[:10]:
    print(f"   - {t}")
if len(classified_as_tags) > 10:
    print(f"   ... and {len(classified_as_tags) - 10} more")

# ============================================================================
# STEP 3: Prepare images for Zenflows (matches GUI prepFilesForZenflows IFile)
# ============================================================================
# The GUI's IFile interface (from lib/types.ts) requires these fields:
#   - name: string (filename)
#   - description: string (same as filename in GUI)
#   - extension: string (file extension without dot)
#   - hash: string (SHA-512 base64url - NOT SHA-256 hex!)
#   - mimeType: string (MIME type)
#   - size: number (file size in bytes)
#
# IMPORTANT: The hash MUST be SHA-512 base64url to match the Zenflows file server!
# The GUI uses zenroom_hash_init("sha512") + base64url encoding.
# ============================================================================

import hashlib
import base64

def base64url_encode(data: bytes) -> str:
    """Encode bytes to base64url (URL-safe base64 without padding)."""
    return base64.urlsafe_b64encode(data).rstrip(b'=').decode('ascii')

def calculate_zenflows_hash(file_path) -> str:
    """Calculate SHA-512 hash of file and return as base64url (matching GUI)."""
    sha512 = hashlib.sha512()
    with open(file_path, 'rb') as f:
        for chunk in iter(lambda: f.read(65536), b''):
            sha512.update(chunk)
    return base64url_encode(sha512.digest())

# Image files are in the assets directory
from pathlib import Path
images_dir = Path('/Users/alcibiade/dyne/if/Interfacer-notebook/assets')

zenflows_images = []
for img_key, img_data in images_clean.items():
    if isinstance(img_data, dict) and (img_data.get('id') or img_data.get('hash') or img_data.get('checksum')):
        filename = img_data.get("fileName", img_data.get("name", img_key))
        extension = filename.split(".")[-1] if "." in filename else "jpg"
        
        # Find the actual image file to calculate correct hash
        img_path = images_dir / filename
        if not img_path.exists():
            for alt_dir in [Path('/Users/alcibiade/dyne/if/Interfacer-notebook/img'), setup_data_dir]:
                alt_path = alt_dir / filename
                if alt_path.exists():
                    img_path = alt_path
                    break
        
        # Calculate SHA-512 base64url hash (matching GUI's createFileHash)
        if img_path.exists():
            file_hash = calculate_zenflows_hash(img_path)
            file_size = img_path.stat().st_size
        else:
            # Fallback to stored hash (won't work for Zenflows but allows testing)
            file_hash = img_data.get("hash", img_data.get("checksum", ""))
            file_size = int(img_data.get("size", 0))
            print(f"   ⚠️ Image file not found: {filename}, using stored hash")
        
        # Get mimeType
        mime_type = img_data.get("mimeType") or img_data.get("contentType") or ""
        if not mime_type:
            ext_lower = extension.lower()
            if ext_lower in ("jpg", "jpeg"):
                mime_type = "image/jpeg"
            elif ext_lower == "png":
                mime_type = "image/png"
            elif ext_lower == "gif":
                mime_type = "image/gif"
            elif ext_lower == "webp":
                mime_type = "image/webp"
            else:
                mime_type = "image/jpeg"
        
        zenflows_images.append({
            "name": filename,
            "description": filename,
            "extension": extension,
            "hash": file_hash,  # SHA-512 base64url!
            "mimeType": mime_type,
            "size": file_size
        })

print(f"\n📸 Images prepared for Zenflows (IFile format): {len(zenflows_images)}")
for img in zenflows_images[:3]:
    print(f"   - {img['name']} ({img['mimeType']}, {img['size']} bytes)")
    print(f"     Hash (SHA-512 base64url): {img['hash'][:40]}...")
if len(zenflows_images) > 3:
    print(f"   ... and {len(zenflows_images) - 3} more")

# ============================================================================
# STEP 4: Build project metadata (matches GUI useProjectCRUD.ts)
# ============================================================================
# NOTE: Images are NOT in metadata! They go to newInventoriedResource.images
# Metadata contains: contributors, licenses, relations, declarations, etc.
# See CREATE_PROJECT mutation structure:
#   - event.resourceMetadata = $metadata (JSON object with form data)
#   - newInventoriedResource.images = $images (IFile array - separate!)
# ============================================================================

project_metadata = {
    "contributors": contributors,
    "licenses": licenses,
    "relations": relations,
    "declarations": declarations,
    "remote": location_remote,
    "design": linked_design,
    "machines": machine_details,
    "materials": material_details,
    "productFilters": product_filters
    # NOTE: images, tags, repo are NOT in metadata - they're separate fields!
}

print(f"\n📋 Project metadata prepared:")
print(f"   Licenses: {len(licenses)}")
print(f"   Contributors: {len(contributors)}")
print(f"   Relations: {len(relations)}")
print(f"   Materials: {len(material_details)}")
print(f"   Design: {linked_design}")

# ============================================================================
# STEP 5: Ensure spec entries exist
# ============================================================================

res_spec_data_local = dict(specs_data)

# Product spec (specProjectProduct from instanceVariables)
res_spec_data_local["locilamp_assembled"] = {
    "id": assembled_spec_id,
    "name": "locilamp_assembled",
    "defaultUnit": unit_each.get("id")
}

# DPP spec (specDpp from instanceVariables)
if SPEC_DPP_ID:
    res_spec_data_local["specDpp"] = {
        "id": SPEC_DPP_ID,
        "name": "specDpp",
        "defaultUnit": unit_each.get("id")
    }

# ============================================================================
# STEP 6: Create DPP Resource (matches GUI CREATE_DPP_RESOURCE mutation)
# ============================================================================

dpp_resource_id = None

if dpp_ulid and SPEC_DPP_ID:
    dpp_res_name = f"DPP for {main_title}"
    dpp_res = {
        "res_ref_id": f'dpp_resource-{random.randint(0, 10000)}',
        "name": dpp_res_name,
        "spec_id": SPEC_DPP_ID
    }
    # GUI passes: dppUlid: JSON.stringify({ dppServiceUlid: dppUlid })
    dpp_resource_metadata = {"dppServiceUlid": dpp_ulid}

    dpp_event_id, _ = create_event(
        users_data['tchibo'],
        action='produce',
        note=f"Digital Product Passport for {main_title}",
        amount=1,
        process=assembly_process,
        res_spec_data=res_spec_data_local,
        new_res=dpp_res,
        metadata=dpp_resource_metadata,
        endpoint=ENDPOINT
    )
    
    dpp_resource_id = dpp_res.get('id')
    print(f"✅ DPP resource created:")
    print(f"   Event ID: {dpp_event_id}")
    print(f"   Resource ID: {dpp_resource_id}")
else:
    print("⚠️ Skipping DPP resource creation (missing DPP ULID or specDpp ID)")

# ============================================================================
# STEP 7: Create the Product (matches GUI CREATE_PROJECT mutation EXACTLY)
# ============================================================================
# The GUI mutation structure from QueryAndMutation.ts:
#   createEconomicEvent(
#     event: { ..., resourceMetadata: $metadata }  <- JSON object
#     newInventoriedResource: { 
#       name: $name, 
#       note: $note, 
#       images: $images,   <- IFile array (NOT in metadata!)
#       repo: $repo, 
#       license: $license 
#     }
#   )
# ============================================================================

# Build the license string (JSON format matching GUI)
license_string = json.dumps(licenses)

# Get Tchibo's location ID for toLocation
tchibo_location_id = locations_data.get("tchibo", {}).get("id") or users_data['tchibo'].get('location_id')

# Use direct GraphQL mutation matching GUI's CREATE_PROJECT exactly
CREATE_PROJECT_MUTATION = '''
mutation CreateProject(
    $name: String!
    $note: String!
    $metadata: JSONObject
    $agent: ID!
    $creationTime: DateTime!
    $location: ID
    $tags: [URI!]
    $resourceSpec: ID!
    $oneUnit: ID!
    $images: [IFile!]
    $repo: String
    $process: ID!
    $license: String!
) {
    createEconomicEvent(
        event: {
            action: "produce"
            provider: $agent
            receiver: $agent
            outputOf: $process
            hasPointInTime: $creationTime
            resourceClassifiedAs: $tags
            resourceConformsTo: $resourceSpec
            resourceQuantity: { hasNumericalValue: 1, hasUnit: $oneUnit }
            toLocation: $location
            resourceMetadata: $metadata
        }
        newInventoriedResource: { 
            name: $name, 
            note: $note, 
            images: $images, 
            repo: $repo, 
            license: $license 
        }
    ) {
        economicEvent {
            id
            resourceInventoriedAs {
                id
                name
                images {
                    hash
                    name
                    mimeType
                }
            }
        }
    }
}
'''

# Prepare variables matching GUI CreateProjectForm.tsx
# NOTE: metadata must be passed as a dict - the json.dumps() in send_signed handles it
# But for JSONObject scalar in GraphQL, we need to check if backend expects string or object
create_project_vars = {
    "name": main_title,
    "note": main_description,  # This shows in Overview tab!
    "metadata": json.dumps(project_metadata),  # JSON string (matching GUI's JSON.stringify)
    "agent": users_data['tchibo']['id'],
    "creationTime": datetime.now().isoformat() + 'Z',
    "location": tchibo_location_id,
    "tags": classified_as_tags,  # resourceClassifiedAs for filtering!
    "resourceSpec": assembled_spec_id,
    "oneUnit": unit_each.get('id') or UNIT_ONE_ID,
    "images": zenflows_images if zenflows_images else None,  # IFile[] to newInventoriedResource!
    "repo": main_link,  # Repository link shown in Overview!
    "process": assembly_process['id'],
    "license": license_string  # License JSON string
}

print(f"\n📤 Creating product with GUI-aligned mutation...")
print(f"   Name: {create_project_vars['name']}")
print(f"   Note (description): {len(create_project_vars['note'])} chars")
print(f"   Repo: {create_project_vars['repo']}")
print(f"   Tags (resourceClassifiedAs): {len(create_project_vars['tags'])} tags")
print(f"   Images (to newInventoriedResource): {len(zenflows_images) if zenflows_images else 0}")
print(f"   License: {license_string[:50]}...")

# Execute the mutation with EdDSA signing
result = send_signed(
    CREATE_PROJECT_MUTATION, 
    create_project_vars,
    users_data['tchibo']['username'],
    users_data['tchibo']['keyring']['eddsa'],
    ENDPOINT
)

if 'errors' in result:
    print(f"❌ Error creating product: {result['errors']}")
    loci_lamp_event = None
    loci_lamp_resource_id = None
else:
    event_data = result.get('data', {}).get('createEconomicEvent', {}).get('economicEvent', {})
    loci_lamp_event = event_data.get('id')
    resource_data = event_data.get('resourceInventoriedAs', {})
    loci_lamp_resource_id = resource_data.get('id')
    
    # Check if images were stored
    stored_images = resource_data.get('images', [])
    
    # Update cur_res for backward compatibility
    res_name = 'locilamp_assembled'
    res_data = resources_data
    res_data[res_name] = {
        "id": loci_lamp_resource_id,
        "name": main_title,
        "spec_id": assembled_spec_id
    }
    cur_res = res_data[res_name]
    
    print(f"\n✅ LOCI LAMP produced with full GUI alignment!")
    print(f"   Event ID: {loci_lamp_event}")
    print(f"   Resource ID: {loci_lamp_resource_id}")
    print(f"   Title: {main_title}")
    print(f"   Repo: {main_link}")
    print(f"   Tags: {len(classified_as_tags)} (for /products filtering)")
    print(f"   Images stored: {len(stored_images)} (from response)")
    print(f"   Note: ✓ (shows in Overview tab)")
    if stored_images:
        for img in stored_images[:3]:
            print(f"      - {img.get('name')} ({img.get('mimeType')})")

# ============================================================================
# STEP 8: Cite DPP Resource (matches GUI CITE_PROJECT mutation)
# ============================================================================

if dpp_resource_id:
    cite_event_id, _ = create_event(
        users_data['tchibo'],
        action='cite',
        note=f"Link DPP {dpp_ulid} to product",
        amount=1,
        process=assembly_process,
        res_spec_data=res_spec_data_local,
        existing_res=dpp_res,
        endpoint=ENDPOINT
    )
    print(f"\n✅ DPP resource cited from product:")
    print(f"   Cite Event ID: {cite_event_id}")
    print(f"   DPP Resource ID: {dpp_resource_id}")

# ============================================================================
# STEP 9: Cite Design Resource (matches GUI linkDesign / CITE_PROJECT mutation)
# ============================================================================
# Products MUST cite their design source - this is REQUIRED for Product type

design_cite_event_id = None
# Use UNIT_ONE_ID which is the standard "one" unit - designs are created with quantity 1 of unitOne
UNIT_ONE_ID = gui_specs.get('unitOne', {}).get('id') if gui_specs else units_data.get('piece', {}).get('id')

if linked_design:
    # Create a cite event to link the design to this product's process
    # IMPORTANT: Must use UNIT_ONE_ID because design was created with that unit
    design_cite_mutation = '''mutation($event:EconomicEventCreateParams!) {
        createEconomicEvent(event: $event) {
            economicEvent { id }
        }
    }'''
    
    cite_vars = {
        'event': {
            'action': 'cite',
            'inputOf': assembly_process['id'],
            'provider': users_data['tchibo']['id'],
            'receiver': users_data['tchibo']['id'],
            'hasPointInTime': datetime.now().isoformat() + 'Z',
            'resourceInventoriedAs': linked_design,
            'resourceQuantity': {'hasNumericalValue': 1, 'hasUnit': UNIT_ONE_ID}
        }
    }
    
    result = send_signed(design_cite_mutation, cite_vars, 
                         users_data['tchibo']['username'], 
                         users_data['tchibo']['keyring']['eddsa'], 
                         ENDPOINT)
    if 'errors' not in result:
        design_cite_event_id = result.get('data', {}).get('createEconomicEvent', {}).get('economicEvent', {}).get('id')
        print(f"\n✅ Design cited from product:")
        print(f"   Cite Event ID: {design_cite_event_id}")
        print(f"   Design Resource ID: {linked_design}")
    else:
        print(f"⚠️ Error citing design: {result['errors']}")
else:
    print(f"\n⚠️ No linked design available - products should cite a design")

# ============================================================================
# STEP 10: Consume Materials (matches GUI MaterialsStep / CONSUME_RESOURCE mutation)
# ============================================================================
# Materials with specMaterial are CONSUMED during production
# Note: Materials were created by locilamp_manufacturer, so they own them
# For production, the manufacturer consumes them into the assembly process

consumed_material_events = []
if materials_used:
    print(f"\n📦 Consuming {len(materials_used)} materials (by manufacturer who owns them)...")
    
    consume_mutation = '''mutation($event:EconomicEventCreateParams!) {
        createEconomicEvent(event: $event) {
            economicEvent { id }
        }
    }'''
    
    # Materials are owned by manufacturer, so manufacturer must consume them
    manufacturer = users_data['locilamp_manufacturer']
    
    for mat_id in materials_used:
        # Look up the material's info including unit from material_resources
        mat_info = next(((name, data) for name, data in material_resources.items() if data.get('id') == mat_id), (None, None))
        mat_name, mat_data = mat_info
        
        # Use the unit the material was created with (stored in material_resources)
        # Fall back to piece unit if not stored
        mat_unit_id = mat_data.get('unit_id', units_data['piece']['id']) if mat_data else units_data['piece']['id']
        
        consume_vars = {
            'event': {
                'action': 'consume',
                'inputOf': assembly_process['id'],
                'provider': manufacturer['id'],
                'receiver': manufacturer['id'],
                'hasPointInTime': datetime.now().isoformat() + 'Z',
                'resourceInventoriedAs': mat_id,
                'resourceQuantity': {'hasNumericalValue': 1, 'hasUnit': mat_unit_id}
            }
        }
        
        result = send_signed(consume_mutation, consume_vars,
                            manufacturer['username'],
                            manufacturer['keyring']['eddsa'],
                            ENDPOINT)
        if 'errors' not in result:
            event_id = result.get('data', {}).get('createEconomicEvent', {}).get('economicEvent', {}).get('id')
            consumed_material_events.append({'material_id': mat_id, 'event_id': event_id, 'name': mat_name})
            print(f"   ✅ Consumed material: {mat_name or mat_id}")
        else:
            print(f"   ⚠️ Error consuming {mat_name or mat_id}: {result['errors']}")
else:
    print(f"\n⚠️ No materials to consume (run Setup notebook to create materials with specMaterial)")

# ============================================================================
# STEP 11: Cite Relations (matches GUI RelationsStep / addRelation)
# ============================================================================

cited_relations = []
if relations:
    print(f"\n🔗 Citing {len(relations)} related resources...")
    
    cite_mutation = '''mutation($event:EconomicEventCreateParams!) {
        createEconomicEvent(event: $event) {
            economicEvent { id }
        }
    }'''
    
    for rel_id in relations:
        cite_vars = {
            'event': {
                'action': 'cite',
                'inputOf': assembly_process['id'],
                'provider': users_data['tchibo']['id'],
                'receiver': users_data['tchibo']['id'],
                'hasPointInTime': datetime.now().isoformat() + 'Z',
                'resourceInventoriedAs': rel_id,
                'resourceQuantity': {'hasNumericalValue': 1, 'hasUnit': unit_each.get('id') or UNIT_ONE_ID}
            }
        }
        
        result = send_signed(cite_mutation, cite_vars,
                            users_data['tchibo']['username'],
                            users_data['tchibo']['keyring']['eddsa'],
                            ENDPOINT)
        if 'errors' not in result:
            event_id = result.get('data', {}).get('createEconomicEvent', {}).get('economicEvent', {}).get('id')
            cited_relations.append({'resource_id': rel_id, 'event_id': event_id})
            print(f"   ✅ Cited relation: {rel_id}")

# ============================================================================
# STEP 12: Store additional form data for reference
# ============================================================================

loci_lamp_metadata = {
    "form_data": {
        "main": {
            "title": main_title,
            "description": main_description,
            "link": main_link,
            "tags": main_tags
        },
        "location": {
            "name": location_name,
            "address": location_address,
            "lat": location_lat,
            "long": location_long,
            "remote": location_remote
        },
        "licenses": licenses,
        "contributors": contributors,
        "relations": relations,
        "declarations": declarations,
        "productFilters": product_filters,
        "machines": machine_details,
        "materials": material_details,
        "linkedDesign": linked_design
    },
    "classifiedAs": classified_as_tags,
    "dpp": {
        "ulid": dpp_ulid,
        "resource_id": dpp_resource_id
    },
    "product": {
        "resource_id": loci_lamp_resource_id,
        "event_id": loci_lamp_event
    },
    "citations": {
        "design": {
            "resource_id": linked_design,
            "event_id": design_cite_event_id
        },
        "dpp": {
            "resource_id": dpp_resource_id,
            "event_id": cite_event_id if dpp_resource_id else None
        },
        "relations": cited_relations
    },
    "consumed": {
        "materials": consumed_material_events,
        "components": consumed_resources
    }
}

print(f"\n" + "="*60)
print("📦 PRODUCT CREATION COMPLETE")
print("="*60)
print(f"\n✅ All GUI form fields populated!")
print(f"   Product: {main_title}")
print(f"   Resource ID: {loci_lamp_resource_id}")
print(f"   DPP ULID: {dpp_ulid}")
print(f"   Design cited: {'Yes' if design_cite_event_id else 'No'}")
print(f"   Materials consumed: {len(consumed_material_events)}")
print(f"   Relations cited: {len(cited_relations)}")
print(f"   Components consumed: {len(consumed_resources)}")
print(f"\n🔗 View product: https://dpp-staging.dyne.im/resource/{loci_lamp_resource_id}")

📋 Form Data Prepared (matching GUI CreateProjectValues):
   Title: LOCI LAMP
   Description: 372 chars
   Repo URL: https://github.com/locilamp/locilamp-v2
   Tags: 8
   Location: Tchibo GmbH Headquarters
   Licenses: 2
   Contributors: 2
   Relations: 3
   Declarations: repairable=yes, recyclable=yes
   Materials: 6
   Machines: 2
   Linked Design: 06EG0TGABRXBS40TENJTTJMZK0
📌 ClassifiedAs tags: 25 tags
   - lamp
   - sustainable
   - cardboard
   - FSC
   - DIY
   - table-lamp
   - design
   - Tchibo
   - m:Laser Cutter
   - m:Paper Cutter
   ... and 15 more

📸 Images prepared for Zenflows (IFile format): 4
   - locilamp_oliverschwartz_DSC05185.jpg (image/jpeg, 891734 bytes)
     Hash (SHA-512 base64url): QeZdQpdkVRm1r5u8awLI8iQBDhUD60xNpOim1ana...
   - locilamp_oliverschwartz_DSC09692.jpg (image/jpeg, 971946 bytes)
     Hash (SHA-512 base64url): MoFeI3CF_egX2WJzhs4M-axRk_MZpDIkUB7U7SU9...
   - locilamp_oliverschwartz_DSC09707.jpg (image/jpeg, 993163 bytes)
     Hash (SHA-512 base64u

In [52]:
# ============================================================================
# UPLOAD IMAGES TO ZENFLOWS FILE SERVER
# ============================================================================
# The GUI uploads image binaries AFTER creating the project via:
#   POST to /zenflows/api/file with FormData where field_name = hash
#
# IMPORTANT: Zenflows uses SHA-512 base64url hash (NOT SHA-256 hex!)
# The hash used for IFile.hash MUST match the hash used for upload form field
# ============================================================================

import hashlib
import base64
from pathlib import Path

# Zenflows file upload URL (same pattern as GUI's NEXT_PUBLIC_ZENFLOWS_FILE_URL)
ZENFLOWS_FILE_URL = ENDPOINT.replace('/api', '/api/file')
print(f"📤 Zenflows File Upload URL: {ZENFLOWS_FILE_URL}")

def base64url_encode(data: bytes) -> str:
    """Encode bytes to base64url (URL-safe base64 without padding)."""
    return base64.urlsafe_b64encode(data).rstrip(b'=').decode('ascii')

def calculate_zenflows_hash(file_path: Path) -> str:
    """Calculate SHA-512 hash of file and return as base64url.
    
    This matches GUI's createFileHash function:
    - Uses SHA-512 (not SHA-256)
    - Returns base64url encoded (not hex)
    """
    sha512 = hashlib.sha512()
    with open(file_path, 'rb') as f:
        for chunk in iter(lambda: f.read(65536), b''):
            sha512.update(chunk)
    return base64url_encode(sha512.digest())

def upload_file_to_zenflows(file_path: Path, file_url: str) -> tuple[bool, str]:
    """Upload a file to Zenflows file server.
    
    Returns (success, hash) tuple.
    
    Matches GUI's uploadFile function:
    - Calculate SHA-512 hash as base64url
    - POST to /zenflows/api/file with FormData(hash: file)
    """
    try:
        file_hash = calculate_zenflows_hash(file_path)
        
        with open(file_path, 'rb') as f:
            files = {file_hash: (file_path.name, f, 'application/octet-stream')}
            response = requests.post(file_url, files=files)
        
        if response.status_code == 200:
            print(f"   ✅ Uploaded: {file_path.name}")
            print(f"      Hash (SHA-512 base64url): {file_hash[:30]}...")
            return True, file_hash
        else:
            print(f"   ⚠️ Upload failed for {file_path.name}: {response.status_code} - {response.text[:100]}")
            return False, file_hash
    except Exception as e:
        print(f"   ❌ Error uploading {file_path.name}: {e}")
        return False, ""

# Image files are in the assets directory
images_dir = Path('/Users/alcibiade/dyne/if/Interfacer-notebook/assets')
uploaded_count = 0

# Store the correct hashes for updating zenflows_images
zenflows_hashes = {}

print(f"\n📸 Uploading images to Zenflows file server...")

for img_key, img_data in images_clean.items():
    if isinstance(img_data, dict):
        filename = img_data.get('fileName', img_data.get('name', f'{img_key}.jpg'))
        # Try to find the image file in assets
        img_path = images_dir / filename
        
        if not img_path.exists():
            # Try alternative locations
            for alt_dir in [Path('/Users/alcibiade/dyne/if/Interfacer-notebook/img'), 
                           setup_data_dir]:
                alt_path = alt_dir / filename
                if alt_path.exists():
                    img_path = alt_path
                    break
        
        if img_path.exists():
            success, file_hash = upload_file_to_zenflows(img_path, ZENFLOWS_FILE_URL)
            if success:
                uploaded_count += 1
                zenflows_hashes[filename] = {
                    'hash': file_hash,
                    'size': img_path.stat().st_size
                }
        else:
            print(f"   ⚠️ Image file not found: {filename}")

print(f"\n✅ Uploaded {uploaded_count} images to Zenflows")

# Update zenflows_images with correct SHA-512 base64url hashes
print(f"\n📝 Updating zenflows_images with correct hashes...")
for i, img in enumerate(zenflows_images):
    if img['name'] in zenflows_hashes:
        old_hash = img['hash']
        new_hash = zenflows_hashes[img['name']]['hash']
        zenflows_images[i]['hash'] = new_hash
        print(f"   Updated {img['name']}:")
        print(f"      Old (SHA-256 hex): {old_hash[:30]}...")
        print(f"      New (SHA-512 b64): {new_hash[:30]}...")

print(f"\n✅ zenflows_images now has correct hashes for Zenflows file server")

📤 Zenflows File Upload URL: https://proxy.dpp-staging-tchibo.dnstest.dyne.org/zenflows/api/file

📸 Uploading images to Zenflows file server...
   ✅ Uploaded: locilamp_oliverschwartz_DSC05185.jpg
      Hash (SHA-512 base64url): QeZdQpdkVRm1r5u8awLI8iQBDhUD60...
   ✅ Uploaded: locilamp_oliverschwartz_DSC09692.jpg
      Hash (SHA-512 base64url): MoFeI3CF_egX2WJzhs4M-axRk_MZpD...
   ✅ Uploaded: locilamp_oliverschwartz_DSC09707.jpg
      Hash (SHA-512 base64url): 5TTzRcKusapFw2YY_iDQpXELvE0SS3...
   ✅ Uploaded: locilamp_oliverschwartz_DSC05218-landscape.jpg
      Hash (SHA-512 base64url): 9GgPwEmN2EkF1vVYdX1ntCpZNadVJm...

✅ Uploaded 4 images to Zenflows

📝 Updating zenflows_images with correct hashes...
   Updated locilamp_oliverschwartz_DSC05185.jpg:
      Old (SHA-256 hex): QeZdQpdkVRm1r5u8awLI8iQBDhUD60...
      New (SHA-512 b64): QeZdQpdkVRm1r5u8awLI8iQBDhUD60...
   Updated locilamp_oliverschwartz_DSC09692.jpg:
      Old (SHA-256 hex): MoFeI3CF_egX2WJzhs4M-axRk_MZpD...
      New (SHA-5

## Save Production Data

In [53]:
# Save production data for reference and tracing
from datetime import datetime

production_data = {
    "dpp_ulid": dpp_ulid,
    "dpp_url": f"{DPP_URL}/dpp/{dpp_ulid}",
    "product": {
        "name": po.get('Product Name', 'LOCI LAMP'),
        "resource_id": loci_lamp_resource_id,
        "event_id": loci_lamp_event,
        "spec_id": assembled_spec_id
    },
    "consumed_components": consumed_resources,
    "production_timestamp": datetime.now().isoformat(),
    "producer": {
        "id": tchibo_id,
        "name": "Tchibo"
    }
}

# Save to JSON file using get_filename pattern
production_file = get_filename('production_data.json', ENDPOINT, USE_CASE)
with open(production_file, 'w') as f:
    json.dump(production_data, f, indent=2)
    
print(f"✅ Production data saved to {production_file}")

✅ Production data saved to use_cases/locilamp/proxy.dpp-staging-tchibo.dnstest.dyne.org%2Fzenflows%2Fapi/production_data.json


## Production Summary

In [54]:
print("=" * 60)
print("🎉 LOCI LAMP PRODUCTION COMPLETE!")
print("=" * 60)
print()
print(f"📦 Product: {po.get('Product Name', 'LOCI LAMP 001')}")
print(f"   Brand: {po.get('Brand Name', 'LoCI')}")
print(f"   Model: {po.get('Model Name', 'LOCI LAMP V 2.0')}")
print()
print(f"🔗 Digital Product Passport:")
print(f"   ULID: {dpp_ulid}")
print(f"   URL: {DPP_URL}/dpp/{dpp_ulid}")
print()
print(f"🏭 Production Details:")
print(f"   Resource ID: {loci_lamp_resource_id}")
print(f"   Producer: Tchibo ({tchibo_id})")
print(f"   Location: Berlin")
print(f"   Components used: {len(consumed_resources)}")
print()
print(f"📸 Images uploaded: {len(images_data)}")
print("=" * 60)

🎉 LOCI LAMP PRODUCTION COMPLETE!

📦 Product: LOCI LAMP
   Brand: Tchibo GmbH
   Model: LOCILAMP V 2.0

🔗 Digital Product Passport:
   ULID: 01KM07CJ7088DMVGWC28X0S1VJ
   URL: https://proxy.dpp-staging-tchibo.dnstest.dyne.org/interfacer-dpp/dpp/01KM07CJ7088DMVGWC28X0S1VJ

🏭 Production Details:
   Resource ID: 06EG0XJNM2ASEV13TDX1RZEQDC
   Producer: Tchibo (06EG0S695QJ78QRCP79B4EYCSC)
   Location: Berlin
   Components used: 3

📸 Images uploaded: 4


## Supply Chain Tracing

Trace the full supply chain of the LOCI LAMP to visualize all components and processes.

In [55]:
# Trace the supply chain from the produced LOCI LAMP
# This uses the trace_query and er_before functions from if_dpp

# trace_query signature is trace_query(id, endpoint)
trace_result = trace_query(loci_lamp_resource_id, ENDPOINT)
print(f"📊 Supply chain trace for {po.get('Product Name', 'LOCI LAMP')}:")
print(f"   Trace depth: {len(trace_result) if trace_result else 0} levels")

# Backward trace using er_before (requires signed request)
trace_me = loci_lamp_resource_id
backtrace = []
visited = set()
er_before(trace_me, users_data['tchibo'], dpp_children=backtrace, depth=0, visited=visited, endpoint=ENDPOINT)
print(f"   Backward trace resources: {len(visited)}")

📊 Supply chain trace for LOCI LAMP:
   Trace depth: 93 levels
id 06EG0XJASTZG3RXPNZKRPRPY60 already in visited in ee_before
id 06EG0THPHPKVQK0E0NGCS2ZYKC already in visited in er_before
id 06EG0THMRHNRNTDMZV47NSM33G already in visited in er_before
id 06EG0THK1GR33DDH0EHX1RNFT8 already in visited in er_before
   Backward trace resources: 46


## Visualization

Generate a Sankey diagram showing the material flow through the supply chain.

In [56]:
# Visualize the DPP and supply chain

print("📊 Generating DPP visualization...")

# Build Sankey data from backward trace
labels = []
sources = []
targets = []
values = []
color_nodes = []
color_links = []
assigned = {}

if backtrace:
    vis_dpp(backtrace[0], count=0, assigned=assigned, labels=labels, targets=targets, sources=sources, values=values, color_nodes=color_nodes, color_links=color_links)
    sources, targets = consol_trace(assigned, sources, targets)
    make_sankey(sources, targets, labels, values, color_nodes, color_links)
else:
    print("No backward trace data available for visualization.")

📊 Generating DPP visualization...


## Completion

The LOCI LAMP has been successfully produced with a Digital Product Passport. The DPP contains:
- Product metadata and specifications
- Material composition and origin
- Sustainability metrics
- Product images
- Manufacturer information
- Links to documentation

The DPP can be accessed at the URL printed above and verified using the ULID.